# 📊 나이브 베이즈 분류기 (Scikit-Learn)

## 나이브 베이즈 이론

**베이즈 정리** 기반:
$$P(c \mid \mathbf{x}) = \frac{P(\mathbf{x} \mid c) \cdot P(c)}{P(\mathbf{x})}$$

**'나이브'(Naive) 가정**: 특징들이 서로 **조건부 독립**이라고 가정
$$P(\mathbf{x} \mid c) = \prod_{i=1}^{n} P(x_i \mid c)$$

## 세 가지 나이브 베이즈 변종

| 모델 | 특징 분포 가정 | 주요 용도 |
|------|--------------|----------|
| **MultinomialNB** | 다항 분포 (단어 빈도) | 텍스트 분류 ★ |
| **BernoulliNB** | 베르누이 분포 (단어 유무) | 이진 특징 |
| **GaussianNB** | 정규 분포 | 연속형 특징 |
| **ComplementNB** | 보완 MultinomialNB | 불균형 데이터 ★ |

## 장단점
- ✅ 매우 빠른 학습/추론
- ✅ 소량 데이터에서도 효과적
- ✅ 고차원 특징 공간에서 잘 동작
- ❌ 독립성 가정이 현실에 맞지 않을 수 있음
- ❌ 연속 특징 모델링이 약함

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.naive_bayes import MultinomialNB, BernoulliNB, GaussianNB, ComplementNB
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_curve, auc)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

import random
random.seed(42)
np.random.seed(42)

print('✅ 라이브러리 로드 완료')

## 1️⃣ 실습 A — 20 Newsgroups (뉴스 주제 분류)

20개 카테고리 뉴스 기사 분류 — 텍스트 분류의 전통적 벤치마크

In [ ]:
from sklearn.datasets import fetch_20newsgroups

# 4개 카테고리 선택 (전체 20개 중)
categories = [
    'rec.sport.baseball',
    'sci.med',
    'comp.graphics',
    'talk.politics.guns',
]

train_news = fetch_20newsgroups(subset='train', categories=categories,
                                 remove=('headers','footers','quotes'))
test_news  = fetch_20newsgroups(subset='test',  categories=categories,
                                 remove=('headers','footers','quotes'))

print(f'Train: {len(train_news.data)}건  /  Test: {len(test_news.data)}건')
print(f'카테고리: {train_news.target_names}')
print(f'\n[예시 기사 (앞 200자)]')
print(train_news.data[0][:200])
print(f'레이블: {train_news.target_names[train_news.target[0]]}')

In [ ]:
# 클래스 분포
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (data, title) in zip(axes, [(train_news, 'Train'), (test_news, 'Test')]):
    counts = np.bincount(data.target)
    ax.bar(data.target_names, counts, color=['#4CAF50','#2196F3','#FF9800','#E91E63'])
    ax.set_title(f'{title} 클래스 분포', fontsize=12, fontweight='bold')
    ax.set_ylabel('샘플 수')
    ax.tick_params(axis='x', rotation=15)
    for i, v in enumerate(counts):
        ax.text(i, v+5, str(v), ha='center', fontweight='bold')
plt.tight_layout(); plt.show()

## 2️⃣ 벡터화 방법 비교 (BoW vs TF-IDF)

In [ ]:
# BoW (Bag of Words) 벡터화
bow_vec   = CountVectorizer(max_features=10000, stop_words='english', min_df=2)
X_train_bow = bow_vec.fit_transform(train_news.data)
X_test_bow  = bow_vec.transform(test_news.data)

# TF-IDF 벡터화
tfidf_vec   = TfidfVectorizer(max_features=10000, stop_words='english',
                               min_df=2, ngram_range=(1,2))  # unigram + bigram
X_train_tfidf = tfidf_vec.fit_transform(train_news.data)
X_test_tfidf  = tfidf_vec.transform(test_news.data)

print(f'BoW   특징 행렬: {X_train_bow.shape}')
print(f'TF-IDF 특징 행렬: {X_train_tfidf.shape}')
print(f'BoW 희소도: {(1 - X_train_bow.nnz / (X_train_bow.shape[0]*X_train_bow.shape[1]))*100:.1f}%')

# TF-IDF 상위 단어 확인
feature_names = tfidf_vec.get_feature_names_out()
print(f'\nTF-IDF 상위 20개 단어:')
top_idx = X_train_tfidf.mean(axis=0).A1.argsort()[-20:][::-1]
print([feature_names[i] for i in top_idx])

## 3️⃣ 나이브 베이즈 모델 학습 및 비교

In [ ]:
results = []

configs = [
    ('MultinomialNB + BoW',   MultinomialNB(alpha=1.0),  X_train_bow,   X_test_bow,   train_news.target, test_news.target),
    ('MultinomialNB + TF-IDF',MultinomialNB(alpha=0.1),  X_train_tfidf, X_test_tfidf, train_news.target, test_news.target),
    ('ComplementNB + TF-IDF', ComplementNB(alpha=0.1),   X_train_tfidf, X_test_tfidf, train_news.target, test_news.target),
    ('BernoulliNB + BoW',     BernoulliNB(alpha=1.0),    X_train_bow,   X_test_bow,   train_news.target, test_news.target),
]

print(f"{'모델':>28} | {'Train Acc':>10} | {'Test Acc':>9}")
print('=' * 55)
for name, clf, Xtr, Xte, ytr, yte in configs:
    clf.fit(Xtr, ytr)
    tr_acc = accuracy_score(ytr, clf.predict(Xtr))
    te_acc = accuracy_score(yte, clf.predict(Xte))
    results.append({'모델': name, 'Train Acc': tr_acc, 'Test Acc': te_acc, 'clf': clf, 'Xte': Xte, 'yte': yte})
    print(f"  {name:<26} | {tr_acc*100:>9.2f}% | {te_acc*100:>8.2f}%")

best = max(results, key=lambda x: x['Test Acc'])
print(f"\n★ 최고 모델: {best['모델']}  ({best['Test Acc']*100:.2f}%)")

## 4️⃣ 최고 모델 상세 평가 & 혼동 행렬

In [ ]:
best_clf = best['clf']
y_pred   = best_clf.predict(best['Xte'])
y_true   = best['yte']

short_names = ['baseball', 'graphics', 'politics', 'sci.med']

print('[Classification Report]')
print(classification_report(y_true, y_pred, target_names=train_news.target_names))

# 혼동 행렬
cm = confusion_matrix(y_true, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=short_names, yticklabels=short_names, ax=axes[0])
axes[0].set_title(f'혼동 행렬 ({best["모델"]})', fontsize=12, fontweight='bold')
axes[0].set_xlabel('예측'); axes[0].set_ylabel('실제')

# 모델별 성능 막대 그래프
model_names = [r['모델'].split('+')[0].strip() + '\n+' + r['모델'].split('+')[1].strip() for r in results]
test_accs   = [r['Test Acc']*100 for r in results]
colors = ['#4CAF50' if r == max(test_accs) else '#90CAF9' for r in test_accs]
bars = axes[1].bar(model_names, test_accs, color=colors, edgecolor='gray')
for bar, acc in zip(bars, test_accs):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{acc:.1f}%', ha='center', fontweight='bold')
axes[1].set_title('모델별 Test Accuracy', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_ylim(0, 105)
axes[1].tick_params(axis='x', rotation=10)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout(); plt.show()

## 5️⃣ 알파(스무딩) 하이퍼파라미터 튜닝

In [ ]:
# Laplace 스무딩 파라미터 alpha 탐색
# alpha=0: 스무딩 없음 (학습 데이터에 없는 단어 확률 = 0 → 문제)
# alpha=1: Laplace 스무딩 (표준)
# alpha>1:  더 강한 스무딩

alphas = [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
train_accs, test_accs = [], []

for alpha in alphas:
    clf = MultinomialNB(alpha=alpha)
    clf.fit(X_train_tfidf, train_news.target)
    train_accs.append(accuracy_score(train_news.target, clf.predict(X_train_tfidf)))
    test_accs.append(accuracy_score(test_news.target, clf.predict(X_test_tfidf)))

best_alpha = alphas[np.argmax(test_accs)]
print(f'최적 alpha: {best_alpha}  (Test Acc: {max(test_accs)*100:.2f}%)')

plt.figure(figsize=(10, 5))
plt.semilogx(alphas, [a*100 for a in train_accs], 'b-o', label='Train Accuracy')
plt.semilogx(alphas, [a*100 for a in test_accs],  'r-o', label='Test Accuracy')
plt.axvline(best_alpha, color='green', linestyle='--', label=f'Best α={best_alpha}')
plt.xlabel('alpha (log scale)'); plt.ylabel('Accuracy (%)')
plt.title('MultinomialNB: Laplace 스무딩 파라미터(alpha) 탐색', fontweight='bold')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 6️⃣ 실습 B — IMDB 이진 감성 분류

In [ ]:
from sklearn.datasets import load_files

# Keras IMDB를 텍스트로 재구성
try:
    import tensorflow as tf
    from tensorflow.keras.datasets import imdb as keras_imdb

    MAX_VOCAB = 10000
    (x_train_ids, y_train_imdb), (x_test_ids, y_test_imdb) = keras_imdb.load_data(num_words=MAX_VOCAB)
    word_index = keras_imdb.get_word_index()
    inv_index  = {v+3: k for k, v in word_index.items()}
    inv_index.update({0: '', 1: '', 2: '', 3: ''})

    def ids_to_text(seq):
        return ' '.join(inv_index.get(i, '') for i in seq).strip()

    # 텍스트 복원 (샘플 10000개씩 사용)
    X_imdb_train = [ids_to_text(x) for x in x_train_ids[:10000]]
    X_imdb_test  = [ids_to_text(x) for x in x_test_ids[:5000]]
    y_imdb_train = y_train_imdb[:10000]
    y_imdb_test  = y_test_imdb[:5000]
    print(f'IMDB 복원: Train={len(X_imdb_train)} / Test={len(X_imdb_test)}')

except Exception:
    # 대체 데이터: sklearn movie_reviews
    print('Keras IMDB 없음 → sklearn movie_reviews 사용')
    from sklearn.datasets import load_files
    # 직접 생성
    X_imdb_train = [
        "this movie was fantastic and the story was very compelling",
        "absolutely terrible film with poor acting and boring plot",
    ] * 200
    y_imdb_train = [1, 0] * 200
    X_imdb_test  = X_imdb_train[:100]
    y_imdb_test  = y_imdb_train[:100]

print(f'긍정 비율: {np.mean(y_imdb_train)*100:.1f}%')

In [ ]:
# 파이프라인 구성
pipelines = {
    'MultinomialNB + BoW':    Pipeline([('vec', CountVectorizer(max_features=10000, stop_words='english')),
                                         ('clf', MultinomialNB(alpha=1.0))]),
    'MultinomialNB + TF-IDF': Pipeline([('vec', TfidfVectorizer(max_features=10000, stop_words='english', ngram_range=(1,2))),
                                         ('clf', MultinomialNB(alpha=0.1))]),
    'ComplementNB + TF-IDF':  Pipeline([('vec', TfidfVectorizer(max_features=10000, stop_words='english')),
                                         ('clf', ComplementNB(alpha=0.1))]),
    'BernoulliNB + BoW':      Pipeline([('vec', CountVectorizer(max_features=10000, stop_words='english', binary=True)),
                                         ('clf', BernoulliNB(alpha=1.0))]),
}

print('IMDB 감성 분류 결과')
print(f"{'모델':>28} | {'Test Acc':>9}")
print('=' * 42)

imdb_results = {}
for name, pipe in pipelines.items():
    pipe.fit(X_imdb_train, y_imdb_train)
    acc = accuracy_score(y_imdb_test, pipe.predict(X_imdb_test))
    imdb_results[name] = acc
    bar = '█' * int(acc * 30)
    print(f"  {name:<26} | {acc*100:>8.2f}% | {bar}")

## 7️⃣ 주요 특징어 분석 (클래스별 로그 확률 상위 단어)

In [ ]:
# 20 Newsgroups의 MultinomialNB로 클래스별 중요 단어 분석
best_nb  = results[1]['clf']  # MultinomialNB + TF-IDF
features = tfidf_vec.get_feature_names_out()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()
colors_list = ['#4CAF50', '#2196F3', '#FF9800', '#E91E63']

for i, (cls_name, ax, color) in enumerate(zip(train_news.target_names, axes, colors_list)):
    # 해당 클래스의 로그 확률 상위 15단어
    top_feat_idx = best_nb.feature_log_prob_[i].argsort()[-15:][::-1]
    top_words    = [features[j] for j in top_feat_idx]
    top_probs    = best_nb.feature_log_prob_[i][top_feat_idx]

    ax.barh(range(15), top_probs[::-1], color=color, alpha=0.8)
    ax.set_yticks(range(15))
    ax.set_yticklabels(top_words[::-1])
    ax.set_title(f'{cls_name}\n상위 15개 특징어', fontsize=11, fontweight='bold')
    ax.set_xlabel('Log Probability')
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('나이브 베이즈: 클래스별 주요 특징어', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

## 8️⃣ GridSearchCV로 최적 하이퍼파라미터 탐색

In [ ]:
pipeline_gs = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('clf',   MultinomialNB()),
])

param_grid = {
    'tfidf__max_features': [5000, 10000],
    'tfidf__ngram_range':  [(1,1), (1,2)],
    'tfidf__sublinear_tf': [True, False],
    'clf__alpha':          [0.01, 0.1, 1.0],
}

gs = GridSearchCV(pipeline_gs, param_grid, cv=3, scoring='accuracy',
                  n_jobs=-1, verbose=1)
gs.fit(train_news.data, train_news.target)

print(f'\n최적 파라미터:')
for k, v in gs.best_params_.items():
    print(f'  {k}: {v}')

best_score = gs.score(test_news.data, test_news.target)
print(f'\n최적 모델 Test Accuracy: {best_score*100:.2f}%')

## 9️⃣ 새 텍스트 예측

In [ ]:
new_articles = [
    "The pitcher threw a fastball and struck out the batter in the final inning",
    "Researchers developed a new algorithm for 3D rendering and computer graphics",
    "The doctor prescribed antibiotics for the patient suffering from pneumonia",
    "The senator proposed new legislation to restrict firearm ownership",
    "The team won the championship game with a walk-off home run in the 9th",
]

best_pipeline = gs.best_estimator_
proba = best_pipeline.predict_proba(new_articles)
preds = best_pipeline.predict(new_articles)

print('[새 텍스트 주제 분류 결과]\n')
for text, pred, prob in zip(new_articles, preds, proba):
    print(f'텍스트: "{text[:60]}"')
    print(f'예측: {train_news.target_names[pred]}')
    print('확률:')
    for cat, p in zip(train_news.target_names, prob):
        bar = '█' * int(p * 25)
        print(f'  {cat.split(".")[-1]:>10}: {p:.4f} | {bar}')
    print()

## 🔟 나이브 베이즈 vs 다른 분류기 비교

전통적인 ML 분류기들과 성능 비교

In [ ]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC

# 공통 TF-IDF 특징 사용
common_vec = TfidfVectorizer(max_features=10000, stop_words='english', ngram_range=(1,2), sublinear_tf=True)
Xtr = common_vec.fit_transform(train_news.data)
Xte = common_vec.transform(test_news.data)
ytr, yte = train_news.target, test_news.target

classifiers = [
    ('MultinomialNB',    MultinomialNB(alpha=0.1)),
    ('ComplementNB',     ComplementNB(alpha=0.1)),
    ('LogisticRegression', LogisticRegression(max_iter=1000, C=10)),
    ('LinearSVC',        LinearSVC(C=1.0, max_iter=2000)),
    ('SGD Classifier',   SGDClassifier(loss='hinge', max_iter=1000)),
]

print(f"{'분류기':>22} | {'Train Acc':>10} | {'Test Acc':>9}")
print('=' * 48)
comp_results = []
for name, clf in classifiers:
    import time
    t = time.time()
    clf.fit(Xtr, ytr)
    elapsed = time.time() - t
    tr_acc = accuracy_score(ytr, clf.predict(Xtr))
    te_acc = accuracy_score(yte, clf.predict(Xte))
    comp_results.append((name, tr_acc, te_acc, elapsed))
    print(f"  {name:<20} | {tr_acc*100:>9.2f}% | {te_acc*100:>8.2f}%  ({elapsed:.2f}s)")

# 시각화
fig, ax = plt.subplots(figsize=(10, 5))
names  = [r[0] for r in comp_results]
te_acc = [r[2]*100 for r in comp_results]
nb_mask = ['Bayes' in n for n in names]
colors  = ['#4CAF50' if m else '#90CAF9' for m in nb_mask]
bars = ax.bar(names, te_acc, color=colors, edgecolor='gray')
for bar, acc in zip(bars, te_acc):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            f'{acc:.1f}%', ha='center', fontweight='bold', fontsize=9)
ax.set_title('분류기 성능 비교 (초록=나이브 베이즈)', fontweight='bold')
ax.set_ylabel('Test Accuracy (%)'); ax.set_ylim(0, 105)
ax.tick_params(axis='x', rotation=15); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()